<a href="https://colab.research.google.com/github/jiafatymah1013-bot/DECODELABS-project4-OCR-text-recognition/blob/main/DECODELABS_project4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!apt-get install -y tesseract-ocr
!pip install pytesseract opencv-python-headless --quiet

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 24 not upgraded.


In [ ]:
"""
Project 4 - Image or Text Recognition (Basic) — OCR Path
DecodeLabs Industrial Training Kit - Batch 2026

Goal: Implement a basic text recognition task using a pre-trained
OCR library (Tesseract via pytesseract), following the DecodeLabs
"Logic Skeleton" for image pre-processing and confidence filtering.

Key Requirements covered:
1. LIBRARY INTEGRATION   -> pytesseract (Tesseract OCR engine wrapper)
2. PRE-PROCESSING        -> Grayscale conversion + Adaptive Thresholding
3. ACCURACY BENCHMARKING -> 80% minimum confidence threshold
4. VISUAL CONFIRMATION   -> Clean, legible extracted text output

NOTE (Google Colab setup):
Tesseract's OCR engine itself is a system binary, not a Python package.
Run this ONCE in a separate Colab cell before running the script:

    !apt-get install -y tesseract-ocr
    !pip install pytesseract opencv-python-headless --quiet
"""

import cv2
import numpy as np
import pytesseract


CONFIDENCE_THRESHOLD = 80  # The 80% Gate, per project spec


def create_sample_image(path="sample_text.png"):
    """
    Generates a simple sample image containing large, bold, high-contrast
    text using OpenCV (rather than a tiny default PIL font), so Tesseract
    can recognize it reliably. This lets the script run end-to-end even
    without the user uploading their own image.
    In Colab you can instead upload a real photo and skip this step.
    """
    img = np.full((300, 900, 3), 255, dtype=np.uint8)  # white canvas

    lines = ["DECODELABS AI PROJECT 4", "OCR RECOGNITION TEST"]
    y = 110
    for line in lines:
        cv2.putText(
            img, line, (40, y),
            cv2.FONT_HERSHEY_SIMPLEX, 1.4, (0, 0, 0), 3, cv2.LINE_AA
        )
        y += 100

    cv2.imwrite(path, img)
    return path


def preprocess_image(image_path):
    """
    PHASE 1: PRE-PROCESSING (The Logic Skeleton)
    Step 1: Grayscale conversion -> collapses 3D RGB into 1D intensity matrix
    Step 2: Gaussian Blur -> removes micro-noise/artifacts
    Step 3: Adaptive Thresholding -> forces pure black-and-white contrast
    """
    image = cv2.imread(image_path)
    if image is None:
        raise FileNotFoundError(f"Could not read image at: {image_path}")

    # Step 1: Grayscale
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # Step 2: Gaussian Blur (smooths noise before thresholding)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)

    # Step 3: Adaptive Thresholding (Otsu-style binary decision per pixel)
    processed = cv2.adaptiveThreshold(
        blurred, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        11, 2
    )

    return image, processed


def run_ocr(processed_image, psm_mode=6):
    """
    PHASE 2: RECOGNITION
    Runs pytesseract with a specified Page Segmentation Mode (PSM).
    PSM 6 = Assume a single uniform block of text (good for clean documents).
    Returns per-word text + confidence scores using image_to_data.
    """
    config = f"--psm {psm_mode}"
    data = pytesseract.image_to_data(
        processed_image, config=config, output_type=pytesseract.Output.DICT
    )
    return data


def filter_by_confidence(data, threshold=CONFIDENCE_THRESHOLD):
    """
    PHASE 3: ACCURACY BENCHMARKING (The 80% Gate)
    Keeps only words whose confidence score meets the minimum threshold.
    Drops low-confidence noise ("confident hallucinations").
    """
    results = []
    for i in range(len(data["text"])):
        word = data["text"][i].strip()
        conf = int(float(data["conf"][i])) if data["conf"][i] != "-1" else -1

        if word and conf >= threshold:
            results.append({
                "text": word,
                "confidence": conf,
                "box": (data["left"][i], data["top"][i], data["width"][i], data["height"][i])
            })
    return results


def draw_annotations(image, results, output_path="output_annotated.png"):
    """
    PHASE 4: VISUAL CONFIRMATION
    Draws bounding boxes + labels for every accepted (>=80% confidence) word.
    """
    annotated = image.copy()
    for r in results:
        x, y, w, h = r["box"]
        label = f"{r['text']} ({r['confidence']}%)"
        cv2.rectangle(annotated, (x, y), (x + w, y + h), (0, 255, 0), 2)
        cv2.putText(annotated, label, (x, max(y - 5, 10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)

    cv2.imwrite(output_path, annotated)
    return output_path


def run_project(image_path=None):
    # If no image is supplied, generate a sample one so the script is self-contained
    if image_path is None:
        image_path = create_sample_image()
        print(f"No image provided -> generated a sample image: {image_path}\n")

    original, processed = preprocess_image(image_path)

    print("=" * 55)
    print(" PROJECT 4: OCR TEXT RECOGNITION ")
    print("=" * 55)
    print("Pre-processing complete: Grayscale -> Blur -> Adaptive Threshold\n")

    ocr_data = run_ocr(processed, psm_mode=6)
    accepted = filter_by_confidence(ocr_data, CONFIDENCE_THRESHOLD)

    if not accepted:
        print(f"No text detected above the {CONFIDENCE_THRESHOLD}% confidence gate.")
        return

    print(f"Detected text (confidence >= {CONFIDENCE_THRESHOLD}%):\n")
    for r in accepted:
        print(f"  '{r['text']}'  -> confidence: {r['confidence']}%")

    full_text = " ".join(r["text"] for r in accepted)
    print(f"\nFull recognized text: \"{full_text}\"")

    output_path = draw_annotations(original, accepted)
    print(f"\nAnnotated output image saved to: {output_path}")


if __name__ == "__main__":
    # To use your OWN image in Colab:
    #   from google.colab import files
    #   uploaded = files.upload()
    #   run_project(list(uploaded.keys())[0])
    #
    # Otherwise this runs on an auto-generated sample image:
    run_project()

No image provided -> generated a sample image: sample_text.png

 PROJECT 4: OCR TEXT RECOGNITION 
Pre-processing complete: Grayscale -> Blur -> Adaptive Threshold

Detected text (confidence >= 80%):

  'DECODELABS'  -> confidence: 91%
  'Al'  -> confidence: 96%
  'PROJECT'  -> confidence: 96%
  '4'  -> confidence: 96%
  'OCR'  -> confidence: 96%
  'RECOGNITION'  -> confidence: 95%
  'TEST'  -> confidence: 96%

Full recognized text: "DECODELABS Al PROJECT 4 OCR RECOGNITION TEST"

Annotated output image saved to: output_annotated.png
